In [ ]:
import pandas as pd
from modules.predictor.data.utils import data_preprocessing

# df_experts1 = pd.read_csv('../../../data/raw/data_experts1.csv')
df_experts1 = data_preprocessing('../../../data/raw/data_experts1.csv', 'expert')
df_experts1

In [ ]:
from pymatgen.io.ase import AseAtomsAdaptor
from ase.visualize import view
from pymatgen.vis.structure_chemview import quick_view
from modules.core.features.symmetries import translate_point_group_to_symmetry_description
from modules.core.features.utils import get_pymatgen_molecule_from_smiles
from pymatgen.symmetry.analyzer import PointGroupAnalyzer
import pandas as pd
import numpy as np

from pymatgen.vis.structure_vtk import StructureVis

from rdkit import Chem
from rdkit.Chem import AllChem, Mol, MolToXYZFile, rdDepictor, rdDistGeom
import matplotlib.pyplot as plt

from modules.core.features.preprocessing import canon_smiles

df = pd.read_csv('../../../data/raw/data_experts1_corrected.csv')
df['canon_smiles'] = [canon_smiles(s) for s in df['smiles'].tolist()]
df = df[['canon_smiles', 'wielkosc_pora teoretyczna']].groupby('canon_smiles').aggregate('mean').reset_index()
pore_sizes = df['wielkosc_pora teoretyczna'].tolist()
smiles = df['canon_smiles'].tolist()

# smiles = df.drop_duplicates(subset=['smiles'])['smiles'].tolist()
# smiles = [canon_smiles(s) for s in smiles]
#
for s, p in zip(smiles, pore_sizes):
    TRANSLATION_TABLE_PATH = "../../../data/symmetries/symmetry_translation.csv"

    molecule = get_pymatgen_molecule_from_smiles(s, save_file=True)
    pga = PointGroupAnalyzer(molecule)
    symmetry_operations = pga.get_symmetry_operations()
    rotational_symmetry = pga.get_rotational_symmetry_number()
    point_group = pga.get_pointgroup()
    equivalent_atoms = pga.get_equivalent_atoms()

    # print(f"Symmetry operations: {symmetry_operations}")
    # print(f"Rotational symmetry number: {rotational_symmetry}")
    print(f"Point group: {point_group}")
    #print(pga.inversion_op)
    #print(f"Equivalent atoms: {equivalent_atoms}")

    symmetry_description = translate_point_group_to_symmetry_description(str(point_group), TRANSLATION_TABLE_PATH)
    print(f"Symmetry description: {symmetry_description}")
    #structure = molecule.get_boxed_structure(23, 23, 23)

    #view(AseAtomsAdaptor().get_atoms(structure))
    # coordinates = []
    # species = []
    #
    # for s in structure:
    #         coordinates.append(s.coords) #cartesian coordinates
    #         species.append(s.specie)
    #
    # print(coordinates)
    # print(species)
    molecule = Chem.MolFromSmiles(s, sanitize=True)
    molecule = Chem.AddHs(molecule)
    rdDepictor.Compute2DCoords(molecule, sampleSeed=42)

    nitrogen_indices = [
            bond.GetEndAtomIdx() if bond.GetBeginAtom().GetSymbol() == "C" and bond.GetEndAtom().GetSymbol() == "N" else bond.GetBeginAtomIdx()
            for bond in molecule.GetBonds()
            if bond.GetBondType() == Chem.BondType.TRIPLE and {"C", "N"} == {bond.GetBeginAtom().GetSymbol(), bond.GetEndAtom().GetSymbol()}
        ]
    if len(nitrogen_indices) == 0:
        nitrogen_indices = list(range(0, molecule.GetNumAtoms()))

    conf = molecule.GetConformer(0)
    coords = conf.GetPositions()

    max_distance = 0
    atom_pos = None

    for j in nitrogen_indices: #range(0, molecule.GetNumAtoms()):
        pos_j = np.array(conf.GetAtomPosition(j))

        # Calculate the distance between the two points
        distance = np.linalg.norm(np.array([0, 0, 0]) - pos_j)

        if distance > max_distance:
            max_distance = distance
            atom_pos = pos_j

    fig = plt.figure(figsize=(6, 4))
    ax = fig.add_subplot(111, projection="3d")
    ax.scatter(coords[:, 0], coords[:, 1], coords[:, 2])
    ax.scatter(0, 0, 0, c='red')
    ax.scatter(atom_pos[0], atom_pos[1], atom_pos[2], c='green')
    plt.show()
    display(Chem.Draw.MolsToImage([molecule]))

    perimeter = max_distance * 2 * 6

    pore_diameter = perimeter / np.pi

    pore_diameter_nm = pore_diameter / 10  # Convert to nanometers
    print(f'Pore size estimated: {pore_diameter_nm}, pore size true: {p}')

    print('--------------------------------------------------------')

In [ ]:
import pandas as pd
import os
maccs_path = '../../../data/fingerprints_maccs/'
data_paths = ['data_experts1', 'data_zhu', 'data_saad', 'data_experts2', 'data_experts3']
dfs = []
for d in data_paths:
    df = pd.read_csv(os.path.join(maccs_path, f'{d}.csv'))
    df.columns = df.columns.str.replace(r"[\[\]>]", "", regex=True)
    df = df.sort_values(by="smiles").reset_index(drop=True)
    dfs.append(df)
df_all = pd.concat(dfs, ignore_index=True).sort_values(by="smiles").reset_index(drop=True)
df_all = df_all.loc[:, ~df_all.eq(df_all.iloc[0]).all()]
df_all.to_csv('maccs_merged.csv')


In [ ]:
df_all

In [ ]:
from modules.core.features.preprocessing import preprocess_target, canon_smiles

df_corrected = pd.read_csv('../../../data/raw/data_experts1_corrected.csv')
column_names_mapping = {
        "zwiazek": "substance",
        "smiles": "smiles",
        "masa_molowa": "molecular_weight",
        "liczba_pierscieni": "number_of_rings",
        "srednia_liczba_N_w_pierscieniu": "average_number_of_N_in_ring",
        "liczba_substratu_w_porze": "number_of_substrates_in_pore",
        "energia Gibbsa": "Gibbs_energy",
        "odleglosc_miedzy_najdalszymi_grupami": "distance_between_furthest_groups",
        "kondensacja": "condensation",
        "piroliza": "pyrolysis",
        "temperatura": "temperature",
        "N_perc_triaz": "N_perc_triaz",
        "wielkosc_pora teoretyczna": "PS",  # Pore Size
        "BET": "SSA",  # BET surface area
        "objetosc": "PV",  # Pore Volume
        "objetosc_mikroporow": "micropore_volume",
        "pojemnosc_3_H2SO4_CV": "capacity_H2SO4_CV",
        "pojemnosc_3_H2SO4_GCD": "capacity_H2SO4_GCD",
        "pojemnosc_3_NaOH_CV": "capacity_NaOH_CV",
        "pojemnosc_3_NaOH_GCD": "capacity_NaOH_GCD",
        "SSSS": "SSSSS",
    }

df_experts = df_corrected.rename(columns=column_names_mapping)
df_experts[["capacity_H2SO4_CV", "capacity_H2SO4_GCD", "capacity_NaOH_CV", "capacity_NaOH_GCD"]] = df_experts[
    ["capacity_H2SO4_CV", "capacity_H2SO4_GCD", "capacity_NaOH_CV", "capacity_NaOH_GCD"]
].replace("~", "", regex=True)
df_experts[["capacity_H2SO4_CV", "capacity_H2SO4_GCD", "capacity_NaOH_CV", "capacity_NaOH_GCD"]] = df_experts[
    ["capacity_H2SO4_CV", "capacity_H2SO4_GCD", "capacity_NaOH_CV", "capacity_NaOH_GCD"]
].astype(float)

# Calculate mean capacity
df_experts["capacity_H2SO4_mean"] = df_experts.loc[:, ["capacity_H2SO4_CV", "capacity_H2SO4_GCD"]].mean(axis=1)
df_experts["capacity_NaOH_mean"] = df_experts.loc[:, ["capacity_NaOH_CV", "capacity_NaOH_GCD"]].mean(axis=1)
df_experts["capacity_mean"] = df_experts.loc[:, ["capacity_H2SO4_mean", "capacity_NaOH_mean"]].mean(axis=1)

targets = ["capacity_H2SO4_CV", "capacity_H2SO4_GCD", "capacity_NaOH_CV", "capacity_NaOH_GCD"]
df_experts["capacity_max"] = preprocess_target(df_experts, targets=targets)

df_experts = df_experts.loc[~df_experts["capacity_max"].isna()].reset_index(drop=True)

df_experts["smiles"] = df_experts["smiles"].apply(canon_smiles)
df_experts['smiles_old'] = df_experts['smiles_old'].apply(canon_smiles)
df_experts.dropna(subset=["smiles", 'smiles_old'], inplace=True)
df_experts = df_experts.rename(columns={'smiles': 'smiles_new', 'smiles_old': 'smiles'})

df_corrected = df_experts[['smiles_new', 'smiles', 'capacity_max']]
df_corrected = df_corrected.groupby(["smiles", "smiles_new"]).max("capacity_max").reset_index()
df_corrected

In [ ]:
df_merged = df_experts1.merge(df_corrected, left_on='smiles', right_on='smiles')
df_merged

In [ ]:
smiles1 = df_experts1['smiles'].tolist()
smiles2 = df_merged['smiles'].tolist()

print(smiles1)

In [ ]:
df_merged = df_merged[['smiles_new', 'capacity_max_y']]
df_merged = df_merged.rename(columns={'smiles_new': 'smiles', 'capacity_max_y': 'capacity_max'})
df_merged

In [ ]:
df_merged.to_csv('../../../data/raw/data_experts1_corrected_matched.csv')

In [ ]:
df_f = pd.read_csv('../../../data/processed_selected_custom_features/data_experts1_corrected_all.csv')
df_merged = df_merged.merge(df_f, left_on='smiles', right_on='smiles')
df_merged

In [ ]:
df_merged.dropna()

In [ ]:
from modules.core.features.preprocessing import canon_smiles
import os

import pandas as pd

from modules.predictor.data.utils import data_preprocessing

# Load data
main_dir = '../../../data'
data_types = {
    'expert': 'data_experts1.csv',
    'expert2': 'data_experts2.csv',
    'expert3': 'data_experts3.csv',
    'zhu': 'data_zhu.csv',
    'saad': 'data_saad.csv',
}
min_smiles = {}
for dtype in data_types:
    print(dtype)
    df = pd.read_csv(os.path.join(main_dir, 'raw', data_types[dtype]))
    df.columns = [x.lower() for x in df.columns]
    df['smiles'] = df['smiles'].apply(canon_smiles)
    df.dropna(subset=['smiles'], inplace=True)
    df.drop_duplicates(subset=['smiles'], inplace=True)
    min_smiles[dtype] = df['smiles'].tolist()

In [ ]:
min_smiles

In [ ]:
for dtype, dlist in min_smiles.items():
    print(dtype, len(dlist))

In [ ]:
for subdir in [s for s in os.listdir(main_dir) if s not in ['sampling', 'raw', 'symmetries', 'processed_all_custom_features']]:
    print(subdir)
    for dtype, dfile in data_types.items():
        df = pd.read_csv(os.path.join(main_dir, subdir, dfile))
        df.columns = [x.lower() for x in df.columns]
        df['smiles'] = df['smiles'].apply(canon_smiles)
        df.dropna(subset=['smiles'], inplace=True)
        df.drop_duplicates(subset=['smiles'], inplace=True)
        smiles_file = df['smiles'].tolist()
        smiles_now = min_smiles[dtype]
        print(len(smiles_now), len(smiles_file), len([x for x in smiles_now if x in smiles_file]))
        new_smiles = [x for x in smiles_now if x in smiles_file]
        min_smiles[dtype] = new_smiles

In [ ]:
for dtype, dlist in min_smiles.items():
    print(dtype, len(dlist))

In [ ]:
for subdir in [s for s in os.listdir(main_dir) if s not in ['sampling', 'raw', 'symmetries', 'processed_all_custom_features']]:
    print(subdir)
    for dtype, dfile in data_types.items():
        df = pd.read_csv(os.path.join(main_dir, subdir, dfile))
        df.columns = [x.lower() for x in df.columns]
        df['smiles'] = df['smiles'].apply(canon_smiles)
        df.dropna(subset=['smiles'], inplace=True)
        df.drop_duplicates(subset=['smiles'], inplace=True)
        df = df[df['smiles'].isin(min_smiles[dtype])]
        print(dtype, subdir, len(df))
        df.to_csv(os.path.join(main_dir, subdir, dfile), index=False)

In [ ]:

paths = ['../../../data/processed_selected_custom_features/data_experts1.csv',
         '../../../data/fingerprints_layered/data_experts1.csv',
         ]
df = [pd.read_csv(data_path) for data_path in paths]
result = df[0]
for data in df[1:]:
    data = data.drop(columns=['capacity_max'])
    result = pd.merge(result, data, on='smiles', how='inner')
# df = pd.concat(df, axis=1).reset_index(drop=True)
result.columns = result.columns.str.replace(r"[\[\]>]", "", regex=True)
result =result.loc[:, ~result.columns.duplicated()].sort_values(by="smiles").reset_index(drop=True)
result
#print(df)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

data_paths = {
    'expert1': '../../../data/processed_selected_custom_features/data_experts1.csv',
    'expert2': '../../../data/processed_selected_custom_features/data_experts2.csv',
    'saad': '../../../data/processed_selected_custom_features/data_saad.csv',
    'zhu': '../../../data/processed_selected_custom_features/data_zhu.csv',
    'expert3': '../../../data/processed_selected_custom_features/data_experts3.csv'
}

# Read the datasets
dfs = {name: pd.read_csv(path) for name, path in data_paths.items()}

# Plot the distributions of capacity_max

plt.figure(figsize=(12, 8))
for name, df in dfs.items():
    print(name, df['capacity_max'].min(), df['capacity_max'].max())
    sns.histplot(df['capacity_max'], kde=True, label=name, bins=30)

plt.title('Distributions of capacity_max in Selected Datasets')
plt.xlabel('capacity_max')
plt.ylabel('Frequency')
plt.legend()
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

data_paths = {
    'expert1': '../../../data/processed_selected_custom_features/data_experts1.csv',
    'expert2': '../../../data/processed_selected_custom_features/data_experts2.csv',
    'saad': '../../../data/processed_selected_custom_features/data_saad.csv',
    'zhu': '../../../data/processed_selected_custom_features/data_zhu.csv',
    'expert3': '../../../data/processed_selected_custom_features/data_experts3.csv'
}

# Read the datasets
dfs = {name: pd.read_csv(path) for name, path in data_paths.items()}

# Combine datasets into a single DataFrame for facet plot
df_combined = pd.concat([df.assign(dataset=name) for name, df in dfs.items()], ignore_index=True)

# Create a violin plot
plt.figure(figsize=(12, 8))
sns.violinplot(x='dataset', y='capacity_max', data=df_combined, cut=0, hue='dataset')
plt.title('Violin Plot of capacity_max in Selected Datasets')
plt.xlabel('Dataset')
plt.ylabel('capacity_max')
plt.show()

# Create a facet plot with each dataset on a separate plot
g = sns.FacetGrid(df_combined, col='dataset', col_wrap=3, height=4, sharex=True, sharey=True, hue='dataset')
g.map(sns.histplot, 'capacity_max', kde=False, bins=30)
g.set_titles(col_template="{col_name}")
g.set_axis_labels('capacity_max', 'Frequency')
plt.show()

# Create a facet plot with each dataset on a separate plot
g = sns.FacetGrid(df_combined, col='dataset', col_wrap=3, height=4, sharex=False, sharey=True, hue='dataset')
g.map(sns.histplot, 'capacity_max', kde=False, bins=30)
g.set_titles(col_template="{col_name}")
g.set_axis_labels('capacity_max', 'Frequency')
plt.show()

In [ ]:
df = pd.concat(dfs, ignore_index=True)
df

In [ ]:
num_bins = 6
bins = pd.qcut(df['capacity_max'], q=num_bins, labels=None)
bins

In [ ]:
num_bins = 6
bins = pd.cut(df['capacity_max'], bins=num_bins, labels=False)
bins

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem

# Define the reaction SMARTS
# This reaction connects two nitrile groups via their nitrogen atoms
reaction_smarts = '[C:1]#[N:2].[C:3]#[N:4]>>[C:1]-[N:2]-[N:4]-[C:3]'
reaction = AllChem.ReactionFromSmarts(reaction_smarts)

# Define the reactant molecules with nitrile groups
mol1 = Chem.MolFromSmiles('N#CC(C#N)=c4ccc(=C(C#N)c3nc(C(C#N)=c1ccc(=C(C#N)C#N)cc1)nc(C(C#N)=c2ccc(=C(C#N)C#N)cc2)n3)cc4')  # Acetonitrile
mol2 = Chem.MolFromSmiles('N#CC(C#N)=c4ccc(=C(C#N)c3nc(C(C#N)=c1ccc(=C(C#N)C#N)cc1)nc(C(C#N)=c2ccc(=C(C#N)C#N)cc2)n3)cc4')  # Propionitrile

# Run the reaction
products = reaction.RunReactants((mol1, mol2))

for _ in range(0):
    product = products[0][0]
    Chem.SanitizeMol(product)
    products = reaction.RunReactants((product, mol1))
product = products[0][0]
display(Chem.Draw.MolToImage(product))

# # Iterate over possible products and display their SMILES
# for product_set in products:
#     for product in product_set:
#         Chem.SanitizeMol(product)
#         print(Chem.MolToSmiles(product))
#         display(Chem.Draw.MolToImage(product))

In [ ]:
import numpy as np
from rdkit.Chem import Draw
from rdkit.Chem import PandasTools
from rdkit import Chem
import pandas as pd
from modules.core.features.utils import canon_smiles


data_paths = {
    # 'expert1': '../../../data/processed_selected_custom_features/data_experts1.csv',
    'expert1_corrected': '../../../data/raw/data_experts1_corrected.csv',
    # 'expert2': '../../../data/processed_selected_custom_features/data_experts2.csv',
    # 'saad': '../../../data/processed_selected_custom_features/data_saad.csv',
    # 'zhu': '../../../data/processed_selected_custom_features/data_zhu.csv',
    # 'expert3': '../../../data/processed_selected_custom_features/data_experts3.csv'
}
for data_name, df_path in data_paths.items():
    df_small = pd.read_csv(df_path)
    print(data_name)

    if data_name == 'expert1_corrected':
        smiles = df_small['smiles'].drop_duplicates().tolist()
    else:
        smiles = df_small['smiles'].drop_duplicates().tolist()
    smiles = [canon_smiles(s) for s in smiles]
    df = pd.DataFrame({'smiles': smiles})
    PandasTools.AddMoleculeColumnToFrame(df, smilesCol='smiles')
    df['id'] = list(range(len(df)))
    for i, s in enumerate(smiles):
        print(i, s)
        mol = Chem.MolFromSmiles(s)
        display(Draw.MolToImage(mol))
    #mols = [Chem.MolFromSmiles(smi) for smi in smiles]
    #display(Draw.MolsToGridImage(mols, molsPerRow=4))
    display(PandasTools.FrameToGridImage(df, legendsCol="id", molsPerRow=4, subImgSize=(400, 400)))
    print('--------------------------------------------')

In [ ]:
import numpy as np
from rdkit.Chem import Draw
from rdkit import Chem

df['bins'] = bins
unique_bins = sorted(np.unique(bins.tolist()))
for b in unique_bins:
    df_small = df[df['bins'] == b]
    print(b, df_small['capacity_max'].min(), df_small['capacity_max'].max(), len(df_small))
    smiles = df_small['smiles'].tolist()
    mols = [Chem.MolFromSmiles(smi) for smi in smiles]
    display(Draw.MolsToGridImage(mols, molsPerRow=4))
    plt.show()

In [ ]:
#point-biserial

import pandas as pd
import numpy as np
from scipy import stats

custom_features = '../../../data/processed_selected_custom_features/data_experts1.csv'
maccs_features = '../../../data/fingerprints_maccs/data_experts1.csv'
df_custom = pd.read_csv(custom_features)
df_custom.sort_values(by='smiles', inplace=True)
df_custom = df_custom[['flatness', 'ps']]
df_maccs = pd.read_csv(maccs_features)
df_maccs.sort_values(by='smiles', inplace=True)
df_maccs.drop(columns=['capacity_max', 'smiles'], inplace=True)

correlation_matrix = pd.DataFrame(index=df_custom.columns, columns=df_maccs.columns)
for i, col1 in enumerate(df_custom.columns):
    for j, col2 in enumerate(df_maccs.columns):
        correlation_matrix.iloc[i, j] = stats.pointbiserialr(df_maccs[col2], df_custom[col1]).correlation

correlation_matrix.T


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
data_names = ['experts1', 'experts2', ]#'experts3', 'saad', 'zhu']
custom_features = [f'../../../data/processed_selected_custom_features/data_{dname}.csv' for dname in data_names]
maccs_features = [f'../../../data/fingerprints_maccs/data_{dname}.csv' for dname in data_names]
df_custom = pd.concat([pd.read_csv(cf) for cf in custom_features], ignore_index=True)
df_custom.sort_values(by='smiles', inplace=True)
df_custom = df_custom[['flatness', 'ps']]
df_maccs = pd.concat([pd.read_csv(cf) for cf in maccs_features], ignore_index=True)
df_maccs.sort_values(by='smiles', inplace=True)
df_maccs.drop(columns=['capacity_max', 'smiles'], inplace=True)
df_maccs.describe()
correlation_matrix = pd.DataFrame(index=df_custom.columns, columns=df_maccs.columns)
for i, col1 in enumerate(df_custom.columns):
    for j, col2 in enumerate(df_maccs.columns):
        correlation_matrix.iloc[i, j] = stats.pointbiserialr(df_maccs[col2], df_custom[col1]).correlation
#
correlation_matrix.T

In [ ]:
correlation_matrix = pd.DataFrame(index=df_custom.columns, columns=df_maccs.columns)
for i, col1 in enumerate(df_custom.columns):
    for j, col2 in enumerate(df_maccs.columns):
        correlation_matrix.iloc[i, j] = stats.spearmanr(df_maccs[col2], df_custom[col1]).correlation
#
correlation_matrix.T

In [ ]:
f_to_remove = []
for i, col in enumerate(df_maccs.columns):
    if df_maccs[col].max() == df_maccs[col].min():
        f_to_remove.append(col)
f_to_remove

In [ ]:
new_dir = '../../../data/selected_maccs'
import os
os.makedirs(new_dir, exist_ok=True)
for data_name in data_names:
    print(data_name)
    df = pd.read_csv(f'../../../data/fingerprints_maccs/data_{data_name}.csv')
    print(df.head())
    df.drop(columns=f_to_remove, inplace=True)
    df.to_csv(f'{new_dir}/data_{data_name}.csv', index=False)